<a href="https://colab.research.google.com/github/vifirsanova/llm-tutorial/blob/main/mas/orchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain-openai langchain-core -qq

from google.colab import userdata

YANDEX_CLOUD_FOLDER = userdata.get('YANDEX_CLOUD_FOLDER')
YANDEX_CLOUD_API_KEY = userdata.get("YANDEX_API_KEY")
YANDEX_CLOUD_MODEL = "gpt-oss-120b/latest"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.4 MB/s eta 0:00:00


## Последовательная оркестрация (Sequential Chain)


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, List

# Подключение к Yandex GPT
llm = ChatOpenAI(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    temperature=0.7
)

# Определяем состояние агента
class SimpleState(TypedDict):
    task: str           # задача
    research: str       # результат исследования
    final_answer: str   # финальный ответ

# Узел 1: Исследование
def research_node(state: SimpleState):
    """Анализируем задачу"""
    prompt = f"Кратко проанализируй задачу: {state['task']}. Выдели 2 ключевых аспекта."
    response = llm.invoke(prompt)
    return {"research": response.content}

# Узел 2: Формирование ответа
def answer_node(state: SimpleState):
    """Генерируем ответ на основе исследования"""
    prompt = f"""На основе этого анализа: {state['research']}
    Дай чёткий ответ на задачу: {state['task']}
    Ответ должен быть практичным и конкретным."""
    response = llm.invoke(prompt)
    return {"final_answer": response.content}

# Строим последовательную цепочку
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(SimpleState)
workflow.add_node("research", research_node)
workflow.add_node("answer", answer_node)
workflow.add_edge(START, "research")
workflow.add_edge("research", "answer")
workflow.add_edge("answer", END)

app = workflow.compile()

# Запускаем
result = app.invoke({
    "task": "Как эффективно выучить новый язык программирования за 2 недели?",
    "research": "",
    "final_answer": ""
})

print("=== ФИНАЛЬНЫЙ ОТВЕТ ===")
print(result["final_answer"])

=== ФИНАЛЬНЫЙ ОТВЕТ ===
**Как выучить новый язык программирования за 2 недели — практический план**  

> **Цель:** к концу 14‑го дня уметь писать небольшие самостоятельные программы, разбираться в основных синтаксических конструкциях и быстро находить нужную информацию в документации.

---

## 1️⃣ Подготовка (0 день)

| Что | Как |
|-----|-----|
| **Выбрать язык** | Убедитесь, что у вас установлен компилятор/интерпретатор и менеджер пакетов. |
| **Собрать «ядро»** | Список тем, которые действительно нужны для практики: <br>• Типы данных, литералы <br>• Операторы, ветвления, циклы <br>• Функции/методы (включая параметры и возврат) <br>• Структуры данных (массив/список, словарь/множество, набор) <br>• Работа с вводом‑выводом и файлами <br>• Основы ООП (если язык её поддерживает) <br>• Пакетный менеджер и сборка/запуск проекта |
| **Создать «cheat‑sheet»** | Таблица‑одностраничник: синтаксис каждой темы + минимальный пример. Делайте её сразу в блокноте/Notion. |
| **Зарегистрироваться** |

## Роутер (условная оркестрация с выбором пути)

In [4]:
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# Подключение к Yandex GPT
llm = ChatOpenAI(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    temperature=0.7
)

# Определяем состояние
class RouterState(TypedDict):
    question: str           # вопрос пользователя
    category: str           # категория вопроса
    answer: str             # ответ

# Узел 1: Классификация (роутер)
def classify_node(state: RouterState):
    """Определяем тип вопроса"""
    prompt = f"""Определи категорию вопроса (только одно слово):
    - 'tech' если про программирование, компьютеры, IT
    - 'business' если про бизнес, маркетинг, стартапы
    - 'other' для всего остального

    Вопрос: {state['question']}
    Категория:"""

    category = llm.invoke(prompt).content.strip().lower()
    return {"category": category}

# Узел 2: Технический ответ
def tech_answer(state: RouterState):
    prompt = f"Дай технический ответ с примерами кода: {state['question']}"
    return {"answer": llm.invoke(prompt).content}

# Узел 3: Бизнес-ответ
def business_answer(state: RouterState):
    prompt = f"Дай бизнес-ориентированный ответ с ROI и метриками: {state['question']}"
    return {"answer": llm.invoke(prompt).content}

# Узел 4: Общий ответ
def general_answer(state: RouterState):
    prompt = f"Дай полезный практический ответ: {state['question']}"
    return {"answer": llm.invoke(prompt).content}

# Функция роутинга
def route_question(state: RouterState) -> Literal["tech", "business", "general"]:
    if state["category"] == "tech":
        return "tech"
    elif state["category"] == "business":
        return "business"
    else:
        return "general"

# Строим граф с условным ветвлением
workflow = StateGraph(RouterState)
workflow.add_node("classify", classify_node)
workflow.add_node("tech", tech_answer)
workflow.add_node("business", business_answer)
workflow.add_node("general", general_answer)

workflow.add_edge(START, "classify")
workflow.add_conditional_edges("classify", route_question)
workflow.add_edge("tech", END)
workflow.add_edge("business", END)
workflow.add_edge("general", END)

app = workflow.compile()

# Тестируем с разными вопросами
questions = [
    "Как оптимизировать SQL запрос?",           # tech
    "Как увеличить конверсию сайта?",           # business
    "Как научиться готовить пасту карбонара?"   # general
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"Вопрос: {q}")
    result = app.invoke({"question": q, "category": "", "answer": ""})
    print(f"Категория: {result['category']}")
    print(f"Ответ: {result['answer'][:150]}...")


Вопрос: Как оптимизировать SQL запрос?
Категория: tech
Ответ: **Оптимизация SQL‑запросов – пошаговый чек‑лист + примеры кода**  

Ниже собран набор практических приёмов, которые работают в большинстве популярных ...

Вопрос: Как увеличить конверсию сайта?
Категория: business
Ответ: **Как увеличить конверсию сайта — пошаговый бизнес‑ориентированный план с расчётом ROI и ключевыми метриками**  

---

## 1. Диагностика текущего сост...

Вопрос: Как научиться готовить пасту карбонара?
Категория: other
Ответ: ## Как научиться готовить пасту карбонара: пошаговый практический план  

### 1. Понимание основ (теория + «зачем»)

| Компонент | Почему важен | Что ...
